# Customer-level task with a trained language model

Takes finished runs' tokenizers and language models (nothing is retrained) and tests
them on a task with one label per customer.

- `TASK_FILE = "fraud"`: is this a flagged customer? (flagged at least once in the
  extract). The fraud flag marks customers from their first transfer on, so this is the
  honest fraud task: no time split, but each customer is scored by a model that never saw them.
- Or a file with one row per customer: `customer id, target (0/1)`.

For each setting in `SETTINGS` (run, history window, pooling):

1. Each customer becomes **one embedding**: the model reads their last
   `window` transfers (all accounts) up to their **window end**, pooled into one vector.
2. The same transfers also give a few **simple statistics** (count, amounts,
   counterparties, time span): the baseline the embedding must beat.
3. XGBoost, 5-fold cross-validation over customers: `raw` (statistics),
   `embeddings`, `combined`. ROC-AUC (0.5 = chance) and AP (chance = share of positives).
   Plus `date only`: the window's end date alone as the score (no model), the time check.

**Window end (`WINDOW_END`).** Flagged customers may stop earlier (blocked or closed
accounts): if every window ended at the customer's last transfer, flagged customers' windows
would be older, and a model could tell them apart by period instead of behaviour (notebook 08).
- `"matched"`: positives keep their last transfer. For each positive's end date,
  `NEG_PER_POS` negatives active on that date are kept, and their window ends at their last
  transfer on or before it, as if they stopped there. The other negatives are left out. Same
  share of positives at every date, so `date only` should be close to chance.
- `"last"` (current choice, 2026-11-17): every window ends at the customer's last transfer.
  The time shortcut stays; `date only` shows how much of a score the date alone gives.

**Set `AS_OF`** for a file task. Only transfers *before* the date the target refers to
may be used. With `AS_OF = None` every transfer in the extract is used, which leaks if the
target happened before the end of the extract (e.g. a churn date).

Needs the GPU (like notebook 04). Appends its results to `<runs root>/customer_task.csv`.

In [ ]:
# ======================================================================
# EDIT HERE
# ======================================================================
# One row per setting: (run, history window, pooling). Each run needs a finished
# language model (stage 03) with one sequence per customer; runs without one are skipped.
# Fastest first.
SETTINGS = [
    ("base", 32, "mean"),          # tiny model (~1M weights): the default
    ("base", 64, "mean"),          # longer history
    ("base", 32, "last_txn"),      # other pooling
    *[(run, 32, "mean") for run in [
        "context_2048", "seed_1", "seed_2",
        "drop_calendar", "drop_operational", "drop_counterparty", "drop_movement",
        "field_order_original", "amount_bins_16", "amount_bins_64", "amt_relative",
        "tdif_bins_96", "tail_hash", "tail_other", "arch_gpt2", "arch_qwen2",
        "loss_weighted", "sep_in_loss", "chunk_overlap",
        "model_small", "steps_5000", "model_31m", "steps_10000", "model_large",
        "early_stop", "early_stop_31m", "early_stop_large", "account_order",
        "early_stop_seed_1", "early_stop_seed_2", "pretrain_on_test",
        "text_words", "text_bag", "text_category", "text_hashed", "mlm", "mlm_txn"]],
]
WINDOW_END = "last"              # "matched": negatives' windows end at dates drawn from the
                                 # positives' end dates (removes the time shortcut);
                                 # "last": every window ends at the last transfer (old behaviour)
NEG_PER_POS = 50                 # "matched": negatives kept per positive (lower it if the
                                 # printout says many positives got fewer)
SEED = 0                         # draw of the matched end dates
TASK_FILE = "fraud"              # "fraud": customer flagged at least once in the extract;
                                 # or a .csv/.parquet file: customer id + target (0/1)
ID_COL, TARGET_COL = "customer_id", "target"
AS_OF = None                     # e.g. "2025-01-01": only transfers before this date
# ======================================================================

import datetime as dt
import gc
import json
import sys
from pathlib import Path

import cudf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.decoder_inference import HuggingFaceDecoderInference
from src.experiment import Experiment, runs_root
from src.tokenizer import BankTransactionTokenizer, BankTransferTokenizerPipeline, HistoryWindowEncoder

OUT = runs_root() / "customer_task.csv"

## 1. One embedding and a few statistics per customer

In [ ]:
def load_task(split_dir, cust):
    """One row per customer: ID_COL, TARGET_COL."""
    if TASK_FILE == "fraud":   # flagged at least once, in any period
        y = pd.concat([pd.read_parquet(split_dir / f"{s}.parquet", columns=[cust, "_target"])
                       for s in ("train", "val", "test")]).groupby(cust)["_target"].max()
        return pd.DataFrame({ID_COL: y.index.astype(str), TARGET_COL: y.values})
    task = pd.read_csv(TASK_FILE) if TASK_FILE.endswith(".csv") else pd.read_parquet(TASK_FILE)
    task = task[[ID_COL, TARGET_COL]].dropna().drop_duplicates(ID_COL)
    task[ID_COL] = task[ID_COL].astype(str)
    return task


def window_ends(split_dir, cust, time, task):
    """Date each task customer's window ends (index: customer id as str). See WINDOW_END."""
    t = pd.concat([pd.read_parquet(split_dir / f"{s}.parquet", columns=[cust, time])
                   for s in ("train", "val", "test")], ignore_index=True)
    t[cust] = t[cust].astype(str)
    t[time] = pd.to_datetime(t[time])
    if AS_OF:
        t = t[t[time] < pd.Timestamp(AS_OF)]
    t = t[t[cust].isin(task[ID_COL])]
    span = t.groupby(cust)[time].agg(["min", "max"])
    if WINDOW_END == "last":
        return span["max"]

    # Positives keep their last transfer. For each positive's end date d (random order),
    # NEG_PER_POS negatives still free and active on d (first transfer <= d <= last) get d,
    # and their window ends at their last transfer on or before d, as if they stopped there.
    # Same share of positives at every date, so the end date tells nothing.
    y = task.set_index(ID_COL)[TARGET_COL].reindex(span.index)
    rng = np.random.default_rng(SEED)
    pos_ends = rng.permutation(span.loc[y == 1, "max"].values)
    neg = span[y == 0]
    first, last = neg["min"].values, neg["max"].values
    free = np.ones(len(neg), dtype=bool)
    draw = np.empty(len(neg), dtype=pos_ends.dtype)
    short = 0
    for d in pos_ends:
        cand = np.flatnonzero(free & (first <= d) & (last >= d))
        take = rng.choice(cand, size=min(NEG_PER_POS, len(cand)), replace=False)
        short += len(take) < NEG_PER_POS
        free[take] = False
        draw[take] = d
    draw = pd.DataFrame({cust: neg.index[~free], "draw": draw[~free]})
    near = pd.merge_asof(draw.sort_values("draw"), t[t[cust].isin(draw[cust])].sort_values(time),
                         left_on="draw", right_on=time, by=cust, direction="backward")
    print(f"Matched end dates: {len(pos_ends):,} positives, {len(draw):,} of {len(neg):,} negatives "
          f"kept ({NEG_PER_POS} per positive; {short:,} positives got fewer)")
    return pd.concat([span.loc[y == 1, "max"], near.set_index(cust)[time]])


def last_n(df, cust, time, n):
    df = df.sort_values([cust, time], ascending=[True, False]).reset_index(drop=True)
    return df[df.groupby(cust).cumcount() < n].reset_index(drop=True)


def customer_features(run, window, pooling):
    """Task, simple statistics, one embedding and the window's end (days) per customer,
    from their last `window` transfers up to their window end."""
    P = Experiment.load(runs_root() / run / "experiment.yaml", validate=False).paths()
    pipeline = BankTransferTokenizerPipeline.from_state(json.loads(P["STATE_PATH"].read_text()))
    tokenizer = BankTransactionTokenizer(pipeline)
    schema = pipeline.schema
    cust, time = schema.customer_id, schema.datetime
    assert pipeline.config.sequence_scope == "customer", f"{run}: needs one sequence per customer"
    model_path = Path(json.loads((P["MODEL_DIR"] / "latest_checkpoint.json").read_text())["checkpoint"])
    inference = HuggingFaceDecoderInference(model_path, tokenizer, pooling=pooling,
                                            tokens_per_txn=pipeline.tokens_per_transaction)
    task = load_task(P["SPLIT_DIR"], cust)
    ends = window_ends(P["SPLIT_DIR"], cust, time, task)
    task = task[task[ID_COL].isin(ends.index)]
    end_of = cudf.DataFrame({"_key": ends.index.values, "_end": ends.values})

    # Each task customer's last `window` transfers up to their window end (before AS_OF).
    parts = []
    for split in ("train", "val", "test"):
        df = cudf.read_parquet(str(P["SPLIT_DIR"] / f"{split}.parquet"))
        # Before the matched end dates (WINDOW_END = "last" gives the same result):
        # df = df[df[cust].astype("str").isin(task[ID_COL].tolist())]
        df["_key"] = df[cust].astype("str")
        df = df.merge(end_of, on="_key")   # task customers only, with their window end
        if df[time].dtype.kind != "M":
            df[time] = cudf.to_datetime(df[time])
        df = df[df[time] <= df["_end"].astype(df[time].dtype)].drop(columns=["_key", "_end"])
        if AS_OF:
            df = df[df[time] < np.datetime64(AS_OF)]
        parts.append(last_n(df, cust, time, window))
    hist = last_n(cudf.concat(parts, ignore_index=True), cust, time, window)
    print(f"{hist[cust].nunique():,} of {len(task):,} customers have transfers ({len(hist):,} rows)")

    # One window per customer, ending at their most recent transfer.
    processed = pipeline.preprocess(hist)
    encoder = HistoryWindowEncoder.from_token_frame(
        pipeline.transform(processed), processed, group_cols=pipeline.group_cols,
        vocab=tokenizer.vocab, unk_id=tokenizer.unk_token_id, window=window,
        max_length=HistoryWindowEncoder.required_length(window, pipeline.tokens_per_transaction),
        pad_id=tokenizer.pad_token_id, bos_id=tokenizer.bos_token_id,
        eos_id=tokenizer.eos_token_id, sep_id=tokenizer.sep_token_id,
    )
    last = np.where(np.r_[encoder.positions[1:] == 0, True])[0]
    emb = inference.extract_embeddings_windowed(encoder.select(last), batch_size=256)
    E = pd.DataFrame(emb, index=processed[cust].astype("str").to_pandas().values[last]).add_prefix("e")
    del inference
    gc.collect()
    torch.cuda.empty_cache()

    # Baseline: simple statistics over the same transfers.
    h = hist.to_pandas()
    g = h.groupby(h[cust].astype(str))
    amt = pd.to_numeric(h[schema.amount], errors="coerce").groupby(h[cust].astype(str))
    raw = pd.DataFrame({
        "n_transfers": g.size(),
        "amt_mean": amt.mean(), "amt_max": amt.max(), "amt_std": amt.std(),
        "n_counterparties": g[schema.counterparty_account].nunique(),
        "days_spanned": (g[time].max() - g[time].min()).dt.days,
    })
    # Days between each window's end and the latest window end: for the time check only,
    # never a model input.
    end_days = (h[time].max() - g[time].max()).dt.days
    print(f"Embeddings {E.shape}, statistics {raw.shape}")
    return task, raw, E, end_days

## 2. Does the embedding predict the target?

Read it like notebook 06: `embeddings` above `raw` by more than about 2 × sd means the
language model knows something the statistics do not; `combined` above `raw` means it adds
to them. Look at AP as well as ROC-AUC: with few positives, ROC-AUC looks good easily.

`date only` is the time check: the window's end date alone as the score. Close to 0.5 / chance
means time is no shortcut. If it is high, the other scores partly measure time, not behaviour.

In [ ]:
# 5-fold cross-validation over customers, for each setting.
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from xgboost import XGBClassifier

cv = StratifiedKFold(5, shuffle=True, random_state=0)


def finished(run):
    """True if the run has a finished language model (stage 03)."""
    info = runs_root() / run / "run_info.json"
    return info.exists() and json.loads(info.read_text())["stages"].get("03", {}).get("status") == "done"


def append(new_rows):
    """Add rows to the results file (read and rewrite, so a file with other columns stays readable)."""
    saved = pd.DataFrame(new_rows).assign(task=TASK_FILE, date=dt.date.today().isoformat())
    if OUT.exists():
        before = pd.read_csv(OUT)
        if "window end" not in before:   # rows from before the matched end dates
            before["window end"] = "last"
        saved = pd.concat([before, saved], ignore_index=True)
    saved.to_csv(OUT, index=False)


todo = [s for s in SETTINGS if finished(s[0])]
for run in dict.fromkeys(s[0] for s in SETTINGS if s not in todo):
    print(f"skipped (no finished language model yet): {run}")
rows = []
for run, window, pooling in todo:
    first_row = len(rows)
    print(f"== {run}, {window} transfers, {pooling}")
    task, raw, E, end_days = customer_features(run, window, pooling)
    data = task.set_index(ID_COL).join(raw, how="inner").join(E, how="inner")
    y = data[TARGET_COL].astype(int).values
    print(f"{len(data):,} customers, {y.sum():,} positives ({y.mean():.2%} = AP by chance)\n")
    # Time check: the window's end date alone as the score (no model, either direction).
    days = end_days.reindex(data.index).values
    auc = roc_auc_score(y, days)
    rows.append({"run": run, "window": window, "pooling": pooling, "window end": WINDOW_END,
                 "input": "date only", "customers": len(data), "positive rate": y.mean(),
                 "AUC": max(auc, 1 - auc), "AUC sd": 0.0, "AP": max(average_precision_score(y, days),
                 average_precision_score(y, -days)), "AP sd": 0.0})
    features = {"raw": list(raw.columns), "embeddings": list(E.columns),
                "combined": list(raw.columns) + list(E.columns)}
    for name, cols in features.items():
        model = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, tree_method="hist")
        s = cross_validate(model, data[cols].values, y, cv=cv, scoring=["roc_auc", "average_precision"])
        rows.append({"run": run, "window": window, "pooling": pooling, "window end": WINDOW_END,
                     "input": name,
                     "customers": len(data), "positive rate": y.mean(),
                     "AUC": s["test_roc_auc"].mean(), "AUC sd": s["test_roc_auc"].std(),
                     "AP": s["test_average_precision"].mean(),
                     "AP sd": s["test_average_precision"].std()})
    append(rows[first_row:])   # saved after each setting: a crash later loses nothing
    print(pd.DataFrame(rows[first_row:])[["input", "AUC", "AP"]].round(3).to_string(index=False), "\n")

result = pd.DataFrame(rows)
print(result.round(3).to_string(index=False))
print(f"\nAppended to {OUT}")

# One group of bars per setting: ROC-AUC (0.5 = chance) and AP (chance = positive rate).
result["setting"] = result["run"] + ", " + result["window"].astype(str) + ", " + result["pooling"]
order = [f"{r}, {w}, {p}" for r, w, p in todo]
fig, axes = plt.subplots(1, 2, figsize=(14, 3 + 0.25 * len(todo)))
for ax, metric, chance in ((axes[0], "AUC", 0.5), (axes[1], "AP", result["positive rate"].iloc[0])):
    table = result.pivot(index="setting", columns="input", values=metric).loc[order, ["date only", "raw", "embeddings", "combined"]]
    sd = result.pivot(index="setting", columns="input", values=f"{metric} sd").loc[order, table.columns]
    table.plot.barh(ax=ax, xerr=sd, capsize=2)
    ax.invert_yaxis()
    ax.axvline(chance, color="grey", ls="--", lw=1)
    ax.set_title(f"{metric} (dashed = chance)")
plt.tight_layout()
plt.show()